In [0]:
%sql
CREATE CATALOG IF NOT EXISTS eodp;
CREATE SCHEMA IF NOT EXISTS eodp.bronze;
CREATE SCHEMA IF NOT EXISTS eodp.silver;
CREATE SCHEMA IF NOT EXISTS eodp.gold;

In [0]:
import requests
from src.lakehouse.bronze_transform import bronze_batch  # <--- აი აქედან მოგვაქვს ჩვენი სკრიპტიდან!
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# API მოთხოვნა
response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 41.7151, "longitude": 44.8271, "current": "temperature_2m,wind_speed_10m"},
)
data = response.json()

raw_record = {
    "city": "Tbilisi",
    "temperature_c": float(data["current"]["temperature_2m"]),
    "wind_speed_kmh": float(data["current"]["wind_speed_10m"])
}

# ვიყენებთ ჩვენს სკრიპტში დაწერილ ფუნქციას მეტამონაცემების დასამატებლად
processed_records = bronze_batch([raw_record], "open-meteo")

schema = StructType([
    StructField("city", StringType(), True),
    StructField("temperature_c", DoubleType(), True),
    StructField("wind_speed_kmh", DoubleType(), True),
    StructField("_bronze_source", StringType(), True),
    StructField("_bronze_ingested_at", StringType(), True)
])

bronze_df = spark.createDataFrame(processed_records, schema=schema)
bronze_df.write.mode("append").saveAsTable("eodp.bronze.weather")
print("[OK] Bronze layer-ში ჩაწერა იმპორტირებული ფუნქციით დასრულდა!")

In [0]:
from pyspark.sql import functions as F

# ვიღებთ მონაცემებს bronze ცხრილიდან
bronze = spark.table("eodp.bronze.weather")

# ვასუფთავებთ და ვფილტრავთ იმ სვეტებზე დაყრდნობით, რომლებიც რეალურად გვაქვს
silver_df = (
    bronze
    .dropDuplicates(["city"])
    .filter((F.col("temperature_c") >= -50) & (F.col("temperature_c") <= 55))
)

# ვწერთ Silver Delta ცხრილში
silver_df.write.mode("overwrite").saveAsTable("eodp.silver.weather")

print("[OK] Silver layer წარმატებით დასრულდა და ჩაიწერა eodp.silver.weather ცხრილში!")

In [0]:
from src.spark_jobs.cross_domain_join_logic import join_by_date
from pyspark.sql import functions as F

# ვიღებთ გასუფთავებულ მონაცემებს Silver ფენიდან
silver = spark.table("eodp.silver.weather")

# ვქმნით Gold ფენის ბიზნეს-დონის აგრეგატს (მომდინარეობს Lab 22-ის კონცეფციიდან)
gold_df = silver.groupBy(F.to_date("_bronze_ingested_at").alias("day")).agg(
    F.round(F.avg("temperature_c"), 1).alias("avg_temp")
)

# ვწერთ Gold Delta ცხრილში
gold_df.write.mode("overwrite").saveAsTable("eodp.gold.daily_weather_summary")

print("[OK] Gold layer წარმატებით დასრულდა და ჩაიწერა eodp.gold.daily_weather_summary ცხრილში!")